Реализуйте алгоритм SAC для среды lunar lander

In [ ]:
!pip install swig
!pip install "gymnasium[box2d]"

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random
from torch.distributions import Normal

In [ ]:
GAMMA = 0.99
TAU = 0.005
ALPHA = 0.2
LR_ACTOR = 3e-4
LR_CRITIC = 3e-4
BUFFER_SIZE = 100000
BATCH = 256
RANDOM_STEPS = 10000
TOTAL_STEPS = 200000
TRAIN_START = 1000
UPDATE_PERIOD = 50

# Выбор устройства
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class ActorModel(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.base = nn.Sequential(
            nn.Linear(state_size, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )
        self.mu = nn.Linear(256, action_size)
        self.log_sigma = nn.Linear(256, action_size)
    
    def compute_action(self, state):
        features = F.relu(self.base(state))
        mean = self.mu(features)
        log_sigma = torch.clamp(self.log_sigma(features), -20, 2)
        sigma = torch.exp(log_sigma)
        
        dist = Normal(mean, sigma)
        sample = dist.rsample()
        action_tanh = torch.tanh(sample)
        
        # Преобразование к диапазону среды
        action_scaled = action_tanh * (ACT_HIGH - ACT_LOW) / 2.0 + (ACT_LOW + ACT_HIGH) / 2.0
        
        # Расчет логарифма вероятности
        log_pi = dist.log_prob(sample)
        log_pi -= torch.log(1 - action_tanh.pow(2) + 1e-6)
        log_pi = log_pi.sum(dim=1, keepdim=True)
        
        return action_scaled, log_pi
    
    def get_action(self, state):
        features = F.relu(self.base(state))
        mean = self.mu(features)
        action_tanh = torch.tanh(mean)
        return action_tanh * (ACT_HIGH - ACT_LOW) / 2.0 + (ACT_HIGH + ACT_LOW) / 2.0

In [ ]:
class CriticModel(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.network1 = nn.Sequential(
            nn.Linear(state_size + action_size, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )
        self.network2 = nn.Sequential(
            nn.Linear(state_size + action_size, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )
    
    def evaluate(self, state, action):
        combined = torch.cat([state, action], dim=-1)
        return self.network1(combined), self.network2(combined)

In [ ]:
class MemoryBuffer:
    def __init__(self, capacity):
        self.data = deque(maxlen=capacity)
    
    def store(self, *experience):
        self.data.append(tuple(experience))
    
    def sample(self, batch_size):
        samples = random.sample(self.data, batch_size)
        states, actions, rewards, next_states, dones = map(np.array, zip(*samples))
        
        return (
            torch.tensor(states, dtype=torch.float32),
            torch.tensor(actions, dtype=torch.float32),
            torch.tensor(rewards, dtype=torch.float32).unsqueeze(1),
            torch.tensor(next_states, dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32).unsqueeze(1)
        )
    
    def size(self):
        return len(self.data)

In [ ]:
# Создание среды
env = gym.make("LunarLanderContinuous-v3")

# Параметры среды
STATE_DIM = env.observation_space.shape[0]
ACTION_DIM = env.action_space.shape[0]
ACT_LOW = float(env.action_space.low[0])
ACT_HIGH = float(env.action_space.high[0])

# Создание моделей
actor = ActorModel(STATE_DIM, ACTION_DIM).to(DEVICE)
critic = CriticModel(STATE_DIM, ACTION_DIM).to(DEVICE)
critic_target = CriticModel(STATE_DIM, ACTION_DIM).to(DEVICE)
critic_target.load_state_dict(critic.state_dict())

# Оптимизаторы
actor_optim = optim.Adam(actor.parameters(), lr=LR_ACTOR)
critic_optim = optim.Adam(critic.parameters(), lr=LR_CRITIC)

# Буфер памяти
memory = MemoryBuffer(BUFFER_SIZE)

# Начальное состояние
state, _ = env.reset()
ep_reward = 0
ep_length = 0

In [ ]:
def update_models():
    if memory.size() < BATCH:
        return
    
    # Загрузка батча
    s, a, r, s_next, d = memory.sample(BATCH)
    
    s = s.to(DEVICE)
    a = a.to(DEVICE)
    r = r.to(DEVICE)
    s_next = s_next.to(DEVICE)
    d = d.to(DEVICE)
    
    # Целевые значения
    with torch.no_grad():
        a_next, log_pi_next = actor(s_next)
        q1_next, q2_next = critic_target(s_next, a_next)
        q_next = torch.min(q1_next, q2_next) - ALPHA * log_pi_next
        target = r + GAMMA * (1 - d) * q_next
    
    # Обновление критика
    q1, q2 = critic(s, a)
    critic_loss = F.mse_loss(q1, target) + F.mse_loss(q2, target)
    
    critic_optim.zero_grad()
    critic_loss.backward()
    critic_optim.step()
    
    # Обновление актора
    a_new, log_pi = actor(s)
    q1_new, q2_new = critic(s, a_new)
    actor_loss = (ALPHA * log_pi - torch.min(q1_new, q2_new)).mean()
    
    actor_optim.zero_grad()
    actor_loss.backward()
    actor_optim.step()
    
    # Мягкое обновление целевой сети
    for param, target_param in zip(critic.parameters(), critic_target.parameters()):
        target_param.data.copy_(TAU * param.data + (1 - TAU) * target_param.data)

In [ ]:
for step in range(TOTAL_STEPS):
    # Выбор действия
    if step < RANDOM_STEPS:
        action = env.action_space.sample()
    else:
        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            action = actor.get_action(state_t).cpu().numpy()[0]
    
    # Шаг в среде
    next_state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    
    # Сохранение в память
    memory.store(state, action, reward, next_state, done)
    
    # Обновление состояния
    state = next_state
    ep_reward += reward
    ep_length += 1
    
    # Завершение эпизода
    if done:
        state, _ = env.reset()
        print(f"Step: {step}, Return: {ep_reward:.2f}, Len: {ep_length}")
        ep_reward = 0
        ep_length = 0
    
    # Обновление моделей
    if step >= TRAIN_START and step % UPDATE_PERIOD == 0:
        for _ in range(UPDATE_PERIOD):
            update_models()

Step: 76, Return: -196.97, Len: 77
Step: 174, Return: -93.74, Len: 98
Step: 287, Return: -373.76, Len: 113
Step: 378, Return: -250.96, Len: 91
Step: 477, Return: -141.53, Len: 99
Step: 596, Return: -60.19, Len: 119
Step: 741, Return: -350.00, Len: 145
Step: 907, Return: -378.58, Len: 166
Step: 972, Return: -92.23, Len: 65
Step: 1042, Return: -83.23, Len: 70
Step: 1145, Return: -346.58, Len: 103
Step: 1242, Return: -334.16, Len: 97
Step: 1339, Return: -119.33, Len: 97
Step: 1510, Return: -232.95, Len: 171
Step: 1622, Return: -300.40, Len: 112
Step: 1700, Return: -80.00, Len: 78
Step: 1810, Return: -186.58, Len: 110
Step: 1915, Return: -208.62, Len: 105
Step: 2038, Return: -379.03, Len: 123
Step: 2112, Return: -79.86, Len: 74
Step: 2202, Return: -154.02, Len: 90
Step: 2285, Return: -258.41, Len: 83
Step: 2437, Return: -322.84, Len: 152
Step: 2570, Return: -305.74, Len: 133
Step: 2674, Return: -314.20, Len: 104
Step: 2809, Return: -202.38, Len: 135
Step: 2904, Return: -420.55, Len: 95
Ste